# Traffic Sign Recognition (GTSRB) with a CNN

End-to-end practice project: load data → explore → preprocess (stratified splits) → build a CNN with augmentation → handle class imbalance → train → evaluate properly (per-class metrics, confusion matrix) → analyze errors.

**Dataset:** GTSRB (German Traffic Sign Recognition Benchmark), 43 classes.

**How to get the data in Colab:** easiest route is Kaggle's GTSRB dataset. Steps:
1. Get a `kaggle.json` API token from your Kaggle account (Account → Create New API Token).
2. Upload it when prompted below.
3. The notebook downloads and unzips the dataset automatically.

## 1. Setup & get the data

In [ ]:
!pip install -q kaggle scikit-learn tensorflow matplotlib seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# --- Kaggle auth: paste your API token from Kaggle -> Settings -> API -> Create New Token ---
# Newer Kaggle tokens look like 'KGAT_...' and are used as an env var / access token file (not kaggle.json).
import os

KAGGLE_TOKEN = 'PASTE_YOUR_TOKEN_HERE'  # e.g. 'KGAT_xxxxxxxxxxxxxxxxxxxxxxxx'

!mkdir -p ~/.kaggle
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write(KAGGLE_TOKEN)
!chmod 600 ~/.kaggle/access_token
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
!unzip -q gtsrb-german-traffic-sign.zip -d gtsrb_data

## 2. Explore the data: class distribution

In [ ]:
# This Kaggle version of GTSRB provides Train.csv / Test.csv with image paths + labels
DATA_DIR = 'gtsrb_data'
train_csv = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))
test_csv = pd.read_csv(os.path.join(DATA_DIR, 'Test.csv'))

print(train_csv.shape, test_csv.shape)
train_csv.head()

In [ ]:
# Class distribution 
class_counts = train_csv['ClassId'].value_counts().sort_index()

plt.figure(figsize=(14, 5))
class_counts.plot(kind='bar')
plt.title('Number of training images per class (GTSRB)')
plt.xlabel('Class ID')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f"Most common class: {class_counts.idxmax()} with {class_counts.max()} images")
print(f"Rarest class: {class_counts.idxmin()} with {class_counts.min()} images")
print(f"Imbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x")

In [ ]:
# Peek at a few sample images across different classes
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
sample_classes = np.random.choice(train_csv['ClassId'].unique(), 10, replace=False)

for ax, cls in zip(axes.flatten(), sample_classes):
    row = train_csv[train_csv['ClassId'] == cls].iloc[0]
    img_path = os.path.join(DATA_DIR, row['Path'])
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(f"Class {cls}")
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Preprocess: resize, normalize, stratified split

In [ ]:
IMG_SIZE = 48  # GTSRB images vary in size; 48x48 keeps detail while training fast
NUM_CLASSES = 43

def load_images(df, data_dir, img_size=IMG_SIZE):
    images = []
    labels = []
    for _, row in df.iterrows():
        img_path = os.path.join(data_dir, row['Path'])
        img = Image.open(img_path).convert('RGB').resize((img_size, img_size))
        images.append(np.array(img))
        labels.append(row['ClassId'])
    return np.array(images, dtype=np.float32) / 255.0, np.array(labels)


X_all, y_all = load_images(train_csv, DATA_DIR)
X_test, y_test = load_images(test_csv, DATA_DIR)

print("Train+val shape:", X_all.shape, "Test shape:", X_test.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=42
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

## 4. Build a CNN with data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15, value_range=(0.0, 1.0)),
    layers.RandomContrast(0.15),
], name='augmentation')

def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)

model = build_cnn()
model.summary()

## 5. Handle class imbalance with weighted loss

In [ ]:
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {int(k): float(v) for k, v in zip(np.unique(y_train), class_weights_array)}

print("Sample class weights (class_id: weight):")
for k in list(class_weight_dict.keys())[:5]:
    print(f"  {k}: {class_weight_dict[k]:.2f}")

## 6. Train

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("y_train dtype:", y_train.dtype, "| unique labels:", np.unique(y_train).min(), "to", np.unique(y_train).max())
print("X_train range:", X_train.min(), "to", X_train.max(), "| any NaN:", np.isnan(X_train).any())
print("class_weight_dict key types:", type(list(class_weight_dict.keys())[0]))

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss')
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

In [ ]:
model.save('gtsrb_model.keras')
print("Saved. Now go to the Files panel on the left, right-click gtsrb_model.keras, and Download.")

from google.colab import files
files.download('gtsrb_model.keras')

In [ ]:
# Plot training curves 
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Evaluate properly: per-class metrics + confusion matrix

Overall accuracy is misleading under class imbalance -- a model that ignores rare classes can still post a high accuracy. Precision/recall/F1 per class tell the real story.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nPer-class classification report:")
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Blues', square=True, cbar=True)
plt.title('Confusion Matrix (43 classes)')
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.show()


cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
top_confusions = np.dstack(np.unravel_index(np.argsort(cm_no_diag.ravel())[::-1], cm_no_diag.shape))[0][:10]

print("Top confused class pairs (true -> predicted, count):")
for true_cls, pred_cls in top_confusions:
    count = cm[true_cls, pred_cls]
    if count > 0:
        print(f"  True {true_cls} -> Predicted {pred_cls}: {count} times")

## 8. Analyze errors: look at actual misclassified images

In [ ]:
misclassified_idx = np.where(y_pred != y_test)[0]
print(f"Total misclassified: {len(misclassified_idx)} / {len(y_test)}")

sample_errors = np.random.choice(misclassified_idx, min(10, len(misclassified_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, idx in zip(axes.flatten(), sample_errors):
    ax.imshow(X_test[idx])
    ax.set_title(f"True: {y_test[idx]}\nPred: {y_pred[idx]}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()